# Chapter 14: Scraping JavaScript (Selenium)

In [48]:
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.webdriver import WebDriver
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service as ChromeService

In [16]:
CHROMEDRIVER_PATH = ChromeDriverManager().install()
service = ChromeService(CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service)

driver.get("http://www.python.org")
time.sleep(2)
driver.close()

In [9]:
chrome_options = ChromeOptions()
chrome_options.add_argument("--headless")
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=chrome_options,
)
driver.get("https://pythonscraping.com/pages/javascript/ajaxDemo.html")
time.sleep(3)
print(driver.find_element(By.ID, "content").text)
driver.close()

Here is some important text you want to retrieve!
A button to click!


In [12]:
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=chrome_options,
)
driver.get("https://pythonscraping.com/pages/javascript/ajaxDemo.html")
time.sleep(3)

page_source = driver.page_source
bs = BeautifulSoup(page_source, "lxml")
print(bs.find(id="content").get_text())

driver.close()

Here is some important text you want to retrieve! A button to click!


In [14]:
print(page_source)

<html><head>
<title>Some JavaScript-loaded content</title>
<script src="../js/jquery-2.1.1.min.js"></script>

</head>
<body>
<div id="content">Here is some important text you want to retrieve! <p></p><button id="loadedButton">A button to click!</button></div>

<script>
$.ajax({
    type: "GET",
    url: "loadedContent.php",
    success: function(response){

	setTimeout(function() {
	    $('#content').html(response);
	}, 2000);
    }
  });

function ajax_delay(str){
 setTimeout("str",2000);
}
</script>

</body></html>


In [31]:
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=chrome_options,
)
driver.get("http://pythonscraping.com/pages/javascript/ajaxDemo.html")
try:
    # implicit wait
    element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "loadedButton"))
    )
finally:
    print(driver.find_element(By.ID, "content").text)
    driver.close()

Here is some important text you want to retrieve!
A button to click!


In [49]:
def wait_for_load(driver: WebDriver) -> None:
    elem = driver.find_element(By.TAG_NAME, "html")
    count = 0
    for _ in range(20):
        try:
            elem == driver.find_element(By.TAG_NAME, "html")
        except StaleElementReferenceException:
            return
        time.sleep(0.5)
    print("Timing out after 10 seconds and returning")


chrome_options = ChromeOptions()
chrome_options.add_argument("--headless")
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=chrome_options,
)
driver.get("http://pythonscraping.com/pages/javascript/redirectDemo1.html")
wait_for_load(driver)
print(driver.page_source)
driver.close()

Timing out after 10 seconds and returning
<html><head>
<title>The Destination Page!</title>

</head>
<body>
This is the page you are looking for!

</body></html>


In [50]:
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=chrome_options,
)
driver.get("http://pythonscraping.com/pages/javascript/redirectDemo1.html")
try:
    txt = "This is the page you are looking for!"
    bodyElement = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.XPATH, f"//body[contains(text(), \"{txt}\")]"))
    )
    print(bodyElement.text)
except TimeoutException:
    print("Did not find the element")

This is the page you are looking for!


In [52]:
driver.quit()